# Hidden-state layer tokens + frozen pretrained text embeddings, fused by a small transformer

Image side: the 29 saved Qwen2.5-7B answer-token states are log-sum-exp pooled into 15 layer-window tokens (the best-performing representation so far), standardized per coordinate with source-train statistics, and projected by a shared linear map plus layer-position embeddings.

Text side: a frozen pretrained encoder (BAAI/bge-base-en-v1.5) reads the question and the generated answer as one sentence pair; its token states are computed once and projected into the fusion width.

A 2-layer fusion transformer trained from scratch attends over [CLS] + layer tokens + text tokens with modality embeddings; a linear head on [CLS] predicts BLEURT-correct vs hallucinated. Text is dropped for 30% of training samples (modality dropout) so the hidden-state branch stays strong on its own.

Three runs with the same split, BCE, seed 42, 10 epochs, best-source-validation-AUROC checkpoint, threshold calibration and held-out TriviaQA evaluation: **fused**, **image only**, **text only**. The fused model is also scored with each modality blanked out.

In [ ]:
!git clone https://github.com/brainardphilemon/SALA.git /kaggle/working/SALA
%cd /kaggle/working/SALA
!git checkout de3a98b
!pip install -q -r requirements-kaggle.txt
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from pathlib import Path
SOURCE_NAME = 'sala_qwen25_g14_6src_triviaqa_v1'
domains = ['tqa', 'nq_open', 'sciq', 'popqa', 'web_questions', 'hotpotqa', 'triviaqa']
artifact_dirs = [p for p in Path('/kaggle/input').rglob(SOURCE_NAME) if p.name == SOURCE_NAME and (p / 'generation').is_dir()]
if len(artifact_dirs) != 1: raise RuntimeError(f'Expected one saved artifact set; found {artifact_dirs}')
ARTIFACT_DIR = artifact_dirs[0]
required = [*(ARTIFACT_DIR / 'generation' / d / f for d in domains for f in ('rows.jsonl', 'features.npy', 'manifest.json')), *(ARTIFACT_DIR / 'bleurt' / d / f for d in domains for f in ('scores.jsonl', 'manifest.json'))]
missing = [str(p) for p in required if not p.is_file()]
if missing: raise RuntimeError('Missing artifact files: ' + ', '.join(missing))
print('Saved features:', ARTIFACT_DIR)

In [ ]:
import subprocess, sys
OUTPUT_DIRS = {mode: Path(f'/kaggle/working/sala_g14_6src_frozen_text_fusion_{mode}') for mode in ('fused', 'image', 'text')}
for mode, output_dir in OUTPUT_DIRS.items():
    command = [sys.executable, '/kaggle/working/SALA/sala_vlm_frozen_text.py', '--input-artifact-dir', str(ARTIFACT_DIR), '--output-dir', str(output_dir), '--bleurt-threshold', '0.5', '--epochs-erm', '10', '--seed', '42', '--train-device', 'cuda:0', '--batch-size', '128', '--image', 'logsumexp', '--mode', mode]
    print('Running:', ' '.join(command), flush=True)
    subprocess.run(command, check=True)

In [ ]:
import json, pandas as pd
rows = []
for mode, output_dir in OUTPUT_DIRS.items():
    summary = json.loads((output_dir / 'summary.json').read_text())
    rows.append({'mode': mode, 'best_epoch': summary['best_epoch'],
                 'source_val_auroc': summary['best_source_validation_auroc_during_training'],
                 'source_test_auroc': summary['source_test_metrics']['auroc'],
                 'target_auroc': summary['target_metrics']['auroc'],
                 'target_balanced_accuracy': summary['target_metrics']['balanced_accuracy'],
                 'text_masked_target_auroc': summary.get('text_masked_target_auroc'),
                 'image_masked_target_auroc': summary.get('image_masked_target_auroc')})
display(pd.DataFrame(rows))
display(pd.read_csv(OUTPUT_DIRS['fused'] / 'results.csv').head())